# 3-bob. 3-BOB. MASHINALI O'QITISH XAVFSIZLIK NUQTAI NAZARIDAN
## Amaliy misol: uchta tasniflash algoritmini taqqoslash

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/03-bob/3-1_uchta-tasniflash-algoritmini-taqqoslash.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 3-bob.*

Quyidagi dasturda tarmoq ulanishlarini tavsiflovchi sun'iy ma'lumotlar yaratiladi. Har bir ulanish to'rtta xususiyat bilan tavsiflanadi: ulanish davomiyligi, yuborilgan ma'lumotlar hajmi, muvaffaqiyatsiz kirish urinishlari soni va daqiqadagi ulanishlar soni. Real tarmoqdagi kabi hujumlar umumiy ulanishlarning atigi 5 foizini tashkil etadi. Ma'lumotlarda uchta algoritm o'qitilib, ularning natijalari taqqoslanadi. Dastur amaliy mashg'ulotning A3 va A9 mavzulariga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

rng = np.random.default_rng(42)

In [2]:
# 1. Sun'iy ma'lumotlar: 1900 ta oddiy va 100 ta hujumga oid ulanish
def ulanishlar(soni, hujum):
    if hujum:
        davomiylik = rng.exponential(10, soni)         # soniya
        baytlar    = rng.normal(2500, 1500, soni)      # yuborilgan hajm
        xato_kirish = rng.poisson(2, soni)             # muvaffaqiyatsiz kirishlar
        ulanish_tezligi = rng.normal(16, 8, soni)      # daqiqadagi ulanishlar
    else:
        davomiylik = rng.exponential(30, soni)
        baytlar    = rng.normal(5000, 2000, soni)
        xato_kirish = rng.poisson(0.5, soni)
        ulanish_tezligi = rng.normal(8, 4, soni)
    return np.column_stack([davomiylik, baytlar, xato_kirish, ulanish_tezligi])

X = np.vstack([ulanishlar(1900, False), ulanishlar(100, True)])
y = np.array([0] * 1900 + [1] * 100)        # 0 - oddiy, 1 - hujum
nomlar = ["davomiylik", "baytlar", "xato_kirish", "ulanish_tezligi"]

In [3]:
# 2. O'qitish va sinov to'plamlariga ajratamiz (sinf ulushini saqlagan holda)
X_o, X_s, y_o, y_s = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

In [4]:
# 3. Uchta algoritmni o'qitib, taqqoslaymiz
modellar = {
    "Logistik regressiya": LogisticRegression(max_iter=1000),
    "Qaror daraxti": DecisionTreeClassifier(max_depth=3, random_state=0),
    "Tasodifiy o'rmon": RandomForestClassifier(n_estimators=100, random_state=0),
}
print(f"{'Model':<22}{'Aniqlik':>9}{'To‘liqlik':>10}{'F1':>7}")
for nom, model in modellar.items():
    model.fit(X_o, y_o)
    p = model.predict(X_s)
    print(f"{nom:<22}{precision_score(y_s, p):>9.2f}{recall_score(y_s, p):>10.2f}{f1_score(y_s, p):>7.2f}")

Model                   Aniqlik To‘liqlik     F1
Logistik regressiya        0.85      0.57   0.68
Qaror daraxti              0.76      0.43   0.55


Tasodifiy o'rmon           0.82      0.60   0.69


In [5]:
# 4. Tasodifiy o'rmon qaysi xususiyatlarga ko'proq tayangan?
rf = modellar["Tasodifiy o'rmon"]
for nom, ahamiyat in sorted(zip(nomlar, rf.feature_importances_), key=lambda t: -t[1]):
    print(f"{nom:<16}{ahamiyat:.2f}")

ulanish_tezligi 0.42
xato_kirish     0.22
baytlar         0.19
davomiylik      0.16


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Model                   Aniqlik To‘liqlik     F1
Logistik regressiya        0.85      0.57   0.68
Qaror daraxti              0.76      0.43   0.55
Tasodifiy o'rmon           0.82      0.60   0.69
ulanish_tezligi 0.42
xato_kirish     0.22
baytlar         0.19
davomiylik      0.16
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 3-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.